# Лекция. Группировки, агрегации и трансформации

In [1]:
import numpy as np 
import pandas as pd

Сегодняшняя тема — это, пожалуй, самый важный рубеж во всем курсе. Если `concat` и `apply` — это инструменты для аккуратной сборки и обработки данных, то группировки — это алхимия, превращающая сырые данные в «золото» инсайтов. На этой лекции мы:

- узнаем, что метод `groupby` не меняет данные, а создает «план захвата мира»;
- познакомимся с концепцией split-apply-combine, лежащей в основе группировок;
- рассмотрим варианты упрощённого синтаксиса для несложных агрегаций;
- научимся работать с методом `agg` — универсальным агрегирующим методом;
- применим `transform` для продвинутого feature engeneering.

В конце лекции — большая задача Data Sciebce с почти реальными данными.

## § 1. Концепция Split-Apply-Combine

Сразу два вопроса:

- зачем вообще нужны группировки и агрегации?
- и что это там написано в заголовке параграфа?

Короткий ответ на первый вопрос — чтобы детали не доминировали над общей целью. То есть, вместо того чтобы смотреть на каждый объект (человека/товар/событие) по отдельности, мы объединяем их в осмысленные группы и смотрим на обобщённые показатели.

Например, в торговой сети каждый день каждый магазин продаёт сотни товаров, и у нас есть хоть капля аккуратности, то мы старательно ведем таблицу с полями: дата, магазин, товар, количество, выручка (и т. д., в зависимости от степени нашего занудства). Вопрос: какой магазин самый прибыльный? Без группировки мы будем смотреть на 100500 строк в день и ничего не поймём. Но после группировки по магазинам, сразу же получим нужный нам ответ (для этого нужно будет провести агрегацию при помощи суммирования, но об этом — чуть позже). Дугой вопрос: а какая категория товаров лучше всего продаётся по вторникам (по пятницам — понятно, что это пиво)? Тут нам понадобится группировка уже по двум признакам: день недели + категория, и это тоже совсем несложная задача для аналитика (если он, конечно, владеет приемами группировки и агрегации, а квалифицированный аналитик ими владеет).

Вот для этого и нужны `groupby` и `agg`. 

Теперь по второму вопросу. Концепция Split-Apply-Combine — это база, на которой стоят все группировки в pandas. Хотя следует сказать, что разработчики пакета все-таки не придумывали её с нуля: они просто реализовали идею, которую в 2011 году сформулировал Хэдли Уикхемс в статье, где сказал, что любую задачу по группировке данных можно разложить на три шага.

**Split** (разделение) — мы берём исходный датафрейм и режем его на кусочки по какому-то признаку. Например, по школьным классам. Или по магазинам. Или по тому, принимал пациент лекарство или плацебо. В pandas это делает groupby: он не показывает результат, а просто запоминает, кому куда идти.

**Apply** (применение) — к каждому кусочку мы применяем какую-то операцию. Что угодно: сумму, среднее, максимум, подсчёт количества и т. д (а можно и свою собственную функцию сначала определить, а потом применить). В результате — получаем для каждой группы какую-то обобщённую характеристику. 

**Combine** (объединение) — результаты из каждого кусочка собираются обратно в единую табличку. Только теперь строк в ней не столько, сколько было исходных записей, а столько, сколько мы использовали групп при выполнении группировки.

Зачем это нам? А затем, что если мы будем понимать split-apply-combine не как команду в pandas, а как образ мысли, то нам  станет легче. Любую задачу из серии «давай посмотрим, а есть ли разница между группами?» мы сможем разложить мысленно на эти три шага. А как это запишется в коде — дело техники.

### Признаки для разделения на группы

Рассмотрим данные об успеваемости наших школьников по аналогии с тем, что мы делали на прошлой лекции:

In [2]:
df = pd.read_csv('data/students.csv')
df

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История
0,A33,Андрей,5,5,4,3,4
1,B15,Серёжа,3,5,5,5,3
2,A95,Настя,5,5,5,4,5
3,A56,Серёжа,4,5,3,2,3
4,A32,Дима,5,4,3,3,4
5,B56,Настя,4,5,5,4,4
6,A41,Миша,4,4,3,3,3
7,B14,Катя,5,4,4,3,4
8,B77,Дима,5,3,5,3,4


Допустим, мы хотим выяснить образовательные параметры по классам (класс A и класс B), это вполне разумное желание. Или сравнить учебные успехи девочек с успехами мальчиков (и это тоже разумное желание). Но есть небольшая проблема: у нас нет соответствующих признаков (а это значит, что их нужно создать). 

Вспомним, что признаки можно создавать при помощи метода `apply` (можно и при помощи других методов, но они работают не всегда, поэтому лучше использовать универсальный `apply`). 

При определении класса школьников достаточно ориентироваться на первую букву их ID. Логику этого процесса выводим во внешнюю функцию `first_letter` после чего применяем метод `apply` к датафрейму `df` (с указанием направления `axis=1`):

In [3]:
def first_letter(row):
    return row['ID'][0]

df['Класс'] = df.apply(first_letter, axis=1)
df

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс
0,A33,Андрей,5,5,4,3,4,A
1,B15,Серёжа,3,5,5,5,3,B
2,A95,Настя,5,5,5,4,5,A
3,A56,Серёжа,4,5,3,2,3,A
4,A32,Дима,5,4,3,3,4,A
5,B56,Настя,4,5,5,4,4,B
6,A41,Миша,4,4,3,3,3,A
7,B14,Катя,5,4,4,3,4,B
8,B77,Дима,5,3,5,3,4,B


Что касается гендерного разделения, все не так просто.

>#### Вопрос
У нас есть только имена школьников. Как определить их пол по имени?

Кривенький, но все-таки работающий подход состоит в том, чтобы вручную составить списки мужских и женских имен. Да, этот подход нельзя назвать изящным. Но мы сегодня занимаемся группировками и агрегациями, а не гендерными вопросами.

Пишем вспомогательную функцию и применяем `apply`:

In [4]:
def gender(row):
    male_names = ['Вася', 'Петя', 'Коля', 'Миша', 'Дима', 'Серёжа', 'Андрей', 'Лёша']
    female_names = ['Катя', 'Маша', 'Даша', 'Настя', 'Оля', 'Лена', 'Ира', 'Юля']
    if row['Имя'] in male_names:
        return 'М'
    elif row['Имя'] in female_names:
        return 'Ж'
    else:
        return ''
    
df['Пол'] = df.apply(gender, axis=1)
df

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
0,A33,Андрей,5,5,4,3,4,A,М
1,B15,Серёжа,3,5,5,5,3,B,М
2,A95,Настя,5,5,5,4,5,A,Ж
3,A56,Серёжа,4,5,3,2,3,A,М
4,A32,Дима,5,4,3,3,4,A,М
5,B56,Настя,4,5,5,4,4,B,Ж
6,A41,Миша,4,4,3,3,3,A,М
7,B14,Катя,5,4,4,3,4,B,Ж
8,B77,Дима,5,3,5,3,4,B,М


## § 2. Метод `groupby` — план по захвату мира

Итак, мы подготовили данные, добавили колонки «Класс» и «Пол» и теперь готовы к группировке. Но здесь всех новичков поджидает главный сюрприз pandas.

Когда мы выполняем команду:

In [5]:
grouped_by_class = df.groupby('Класс')
grouped_by_class

— то в результате мы не видим никакой красивой таблички, а видим какую-то неудобоваримую запись. Что это? Где группы? Где результат? 

И когда выполняем команду:

In [6]:
grouped_by_sex = df.groupby('Пол')
grouped_by_sex

— тоже не видим ничего хорошего. Такое ощущение, что нас обманули.

Но не стоит отчаиваться. На самом деле, `groupby` — это отложенное действие. Пандас не делает лишней работы: он просто просмотрел таблицу, составил список, например,  «кто в каком классе учится» (или, во втором случае, «кто из них мальчик, и кто девочка») и замер в ожидании наших дальнейших указаний. Это и есть тот самый этап Split из теории Хэдли Уикхема.

### Что под капотом у `groupby`?

Чтобы развеять мистику, давайте заглянем под капот. У этих объектов, созданных нами выше, есть свойство `groups`. Например:

In [7]:
grouped_by_class.groups

{'A': [0, 2, 3, 4, 6], 'B': [1, 5, 7, 8]}

Или:

In [8]:
grouped_by_sex.groups

{'Ж': [2, 5, 7], 'М': [0, 1, 3, 4, 6, 8]}

Это обычные питоновские словари, где ключи — это названия групп, а значения — списки индексов строк, которые в эти группы попали. Как обычно, мы можем обратиться к словарю по ключу:

In [9]:
grouped_by_sex.groups['Ж']

Int64Index([2, 5, 7], dtype='int64')

Или:

In [10]:
grouped_by_class.groups['A']

Int64Index([0, 2, 3, 4, 6], dtype='int64')

Но самое интересное происходит, когда мы начинаеми перебирать эти объекты в цикле (да: `grouped_by_sex` и `grouped_by_class` — это итерируемые объекты).

### Анатомия цикла: переменные `name` и `group`

Объект, который получается после применения метода `groupby`, при итерации по нему выдает пару значений на каждом шаге. Поэтому на вход циклу подают пару счетчиков и обычно их называют согласно их семантике — 1) имя группы и 2) собственно группа:

In [11]:
for name, group in grouped_by_class:
    display(name, group)

'A'

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
0,A33,Андрей,5,5,4,3,4,A,М
2,A95,Настя,5,5,5,4,5,A,Ж
3,A56,Серёжа,4,5,3,2,3,A,М
4,A32,Дима,5,4,3,3,4,A,М
6,A41,Миша,4,4,3,3,3,A,М


'B'

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
1,B15,Серёжа,3,5,5,5,3,B,М
5,B56,Настя,4,5,5,4,4,B,Ж
7,B14,Катя,5,4,4,3,4,B,Ж
8,B77,Дима,5,3,5,3,4,B,М


In [12]:
for name, group in grouped_by_sex:
    display(group)

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
2,A95,Настя,5,5,5,4,5,A,Ж
5,B56,Настя,4,5,5,4,4,B,Ж
7,B14,Катя,5,4,4,3,4,B,Ж


,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
0,A33,Андрей,5,5,4,3,4,A,М
1,B15,Серёжа,3,5,5,5,3,B,М
3,A56,Серёжа,4,5,3,2,3,A,М
4,A32,Дима,5,4,3,3,4,A,М
6,A41,Миша,4,4,3,3,3,A,М
8,B77,Дима,5,3,5,3,4,B,М


Вот теперь мы наконец-то видим примерно то, что мы ожидали: последовательность групп. Причем, обратите внимание: в отличие от свойства `groups`, которое выдает список номеров строк (что не особенно удобно), цикл возвращает именно датафреймы. Посмотреть на нах глазками бывает очень полезно, особенно при отладке кода группировки.

Отметим, что посмотреть на какую-то конкретную группу при помощи цикла не удастся. Но во-первых, для этого существетстарый добрый метод `loc`:

In [13]:
df.loc[df['Класс'] == 'A']

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
0,A33,Андрей,5,5,4,3,4,A,М
2,A95,Настя,5,5,5,4,5,A,Ж
3,A56,Серёжа,4,5,3,2,3,A,М
4,A32,Дима,5,4,3,3,4,A,М
6,A41,Миша,4,4,3,3,3,A,М


А во-вторых, можно воспользоваться методом `get_group`:

In [14]:
grouped_by_class.get_group('A')

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
0,A33,Андрей,5,5,4,3,4,A,М
2,A95,Настя,5,5,5,4,5,A,Ж
3,A56,Серёжа,4,5,3,2,3,A,М
4,A32,Дима,5,4,3,3,4,A,М
6,A41,Миша,4,4,3,3,3,A,М


И все-таки метод `groupby` нужен не для того, чтобы разглядывать группы. А для чего? — разбираемся дальше.

### Группировка по нескольким признакам

Можно выполнять группировку, используя список из нескольких признаков. Например, группировка одновременно и по полу, и по классу выполняется так:

In [15]:
grouped_by_class_and_sex = df.groupby(['Класс', 'Пол'])
grouped_by_class_and_sex

Опять-таки мы видим лишь, что это какой-то объект, а посмотреть на отдельные группы (сразу на все) можно в цикле:

In [16]:
for name, group in grouped_by_class_and_sex:
    display(name, group)

('A', 'Ж')

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
2,A95,Настя,5,5,5,4,5,A,Ж


('A', 'М')

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
0,A33,Андрей,5,5,4,3,4,A,М
3,A56,Серёжа,4,5,3,2,3,A,М
4,A32,Дима,5,4,3,3,4,A,М
6,A41,Миша,4,4,3,3,3,A,М


('B', 'Ж')

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
5,B56,Настя,4,5,5,4,4,B,Ж
7,B14,Катя,5,4,4,3,4,B,Ж


('B', 'М')

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
1,B15,Серёжа,3,5,5,5,3,B,М
8,B77,Дима,5,3,5,3,4,B,М


Можно поменять последовательность элементов в списке признаков для группировки и снова запустить цикл:

In [17]:
grouped_by_sex_and_class = df.groupby(['Пол', 'Класс'])
for name, group in grouped_by_sex_and_class:
    display(name, group)

('Ж', 'A')

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
2,A95,Настя,5,5,5,4,5,A,Ж


('Ж', 'B')

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
5,B56,Настя,4,5,5,4,4,B,Ж
7,B14,Катя,5,4,4,3,4,B,Ж


('М', 'A')

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
0,A33,Андрей,5,5,4,3,4,A,М
3,A56,Серёжа,4,5,3,2,3,A,М
4,A32,Дима,5,4,3,3,4,A,М
6,A41,Миша,4,4,3,3,3,A,М


('М', 'B')

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
1,B15,Серёжа,3,5,5,5,3,B,М
8,B77,Дима,5,3,5,3,4,B,М


Как и следовало ожидать, мы видим те же самые группы, хотя и в другой последовательности. Логично предположить, что мы получили тот же объект, что был у нас при предыдущем порядке группировки. 

Однако не будем спешить. Давйте посмотрим:

In [18]:
grouped_by_class_and_sex == grouped_by_sex_and_class

False

Вот так. Это не одно и то же.

Дело в том, что метод `groupby` дает не только сведения о том, к какой группе относится тот или иной объект. Он дает еще и сведения о порядке действий. 

Когда мы группируем в проядке `['Класс', 'Пол']`, порядок действий такой: 

1) фиксируем первое значение для `'Класс'` (это `'A'`),
2) при фиксированном классе перебираем все значения `'Пол'` (это сначала `'Ж'`, а потом `'М'`, причем именно в таком порядке, а не наоборот),
3) фиксируем второе значение для `'Класс'` (это `'B'`),
4) при фиксированном классе перебираем все значения `'Пол'`.

А когда группируем в порядке `['Пол', 'Класс']`, порядок действий другой:

1) фиксируем первое значение для `'Пол'` (это `'Ж'`),
2) при фиксированном поле перебираем все значения `'Класс'` (это сначала `'A'`, а потом `'B'`),
3) фиксируем второе значение для `'Пол'` (это `'М'`),
4) при фиксированном поле перебираем все значения `'Класс'`.

Да, в результате смены порядка группировки мы получаем то же самое множество групп. Но мы приходим к тому же самому множеству за счет другой последовательности шагов.

### Резюме по `groupby`

Метод `groupby` расписывает порядок действий и их результаты: в какой последовательности будут выполняться шаги группировки, и какие группы будут при этом получаться. Но сам по себе он еще ничего не делает. Метод `groupby` — это «план по захвату мира», но еще не сам захват.

Захват происходит применением метода `agg`.

## § 3. Метод `agg` — путь джедая и немного сахара

Теперь, когда у нас есть «план захвата» (объект groupby), самое время осуществить захват. В теории Хэдли Уикхема этому соответствуют этапы Apply (применить функцию к каждой группе) и Combine (склеить результаты обратно в таблицу), а в Pandas для этого есть универсальный инструмент — метод `agg` (сокращение от aggregate — агрегировать, собирать в целое).

### Универсальный метод `agg`

Создаем словарь-инструкцию:  

- по алгебре — нужно посчитать среднюю оценку в каждой группе,
- по информатике — средний разброс значений в каждой группе,
- и кроме того — вывести число уникальных имен в каждой группе.

Группируем по позиции `'Пол'`, а то что получилось агрегируем по инструкции `instruction`:

In [19]:
instruction = {
    'Алгебра': 'mean',
    'Информатика': 'mean',
    'Имя': 'nunique'
}

result = df.groupby('Пол').agg(instruction)
display(result)

,Алгебра,Информатика,Имя
Пол,,,
Ж,4.666667,4.666667,2
М,4.333333,3.833333,4


Видим, что девочки чуть-чуть умнее мальчиков. На самом деле, это научный факт: в среднем девочки всегда учатся лучше мальчиков — достаточно посмотреть на успеваемость в любой студенческой группе на любом факультете (хотя, бывают, конечно, и исключения).

Но это нам сейчас не важно (мы же учимся группировкам, а не исследуем гендерные вопросы). А важно вот что: когда мы используем `agg`, Pandas берет на себя всю грязную работу по склеиванию результатов (Combine). Нам не нужно писать циклы и вручную соединять кусочки — на выходе мы получаем готовую, чистенькую таблицу, где индексами стали наши группы.

Понятно, что можно сгруппировать данные не по полу, а по классу:

In [20]:
instruction = {
    'Алгебра': 'mean',
    'Информатика': 'mean',
    'Имя': 'nunique'
}

result = df.groupby('Класс').agg(instruction)
display(result)

,Алгебра,Информатика,Имя
Класс,,,
A,4.60,3.60,5
B,4.25,4.75,4


И после этого сказать, что школьники из класса A учатся лучше школьников из класса B — язык не повернется. Результаты весьма противоречивые: по алгебре — да, лучше, а по информатике — значительно хуже.

Можно сгруппировать данные по двум  признакам:

In [21]:
instruction = {
    'Алгебра': 'mean',
    'Информатика': 'mean',
    'Имя': 'nunique'
}

result = df.groupby(['Класс', 'Пол']).agg(instruction)
display(result)

Алгебра  Информатика  Имя
Класс Пол                           
A     Ж        5.0         5.00    1
      М        4.5         3.25    4
B     Ж        4.5         4.50    2
      М        4.0         5.00    2

И становится понятно, что успеваемость по информатике в классе A такая низкая потому, что в классе A учится много мальчиков, а девочка всего одна (она-то учится как раз хорошо, даже отлично, но не может в одиночку вытащить весь класс). Но мы опять отвлеклись. 

При смене порядка группировки мы получим другую сводную таблицу:

In [22]:
instruction = {
    'Алгебра': 'mean',
    'Информатика': 'mean',
    'Имя': 'nunique'
}

result = df.groupby(['Пол', 'Класс']).agg(instruction)
display(result)

Алгебра  Информатика  Имя
Пол Класс                           
Ж   A          5.0         5.00    1
    B          4.5         4.50    2
М   A          4.5         3.25    4
    B          4.0         5.00    2

Это потому что последовательность действий в этом случае другая, да и логика результатов — тоже.

Теперь допустим, что нас интересует успеваемость по предметам естественнонаучного блока, распределенная по классам , причем, нам нужны сразу несколько параметров: максимальная оценка, минимальная оценка и средняя оценка. При помощи комбинации `groupby` и `agg` сделать это совсем легко, мы просто используем списки ключей:

In [23]:
instruction = {
    'Алгебра': ['max', 'min', 'mean'],
    'Геометрия': ['max', 'min', 'mean'],
    'Информатика': ['max', 'min', 'mean'],
   
}

result = df.groupby('Класс').agg(instruction)
display(result)

Алгебра           Геометрия           Информатика          
          max min  mean       max min  mean         max min  mean
Класс                                                            
A           5   4  4.60         5   4  4.60           5   3  3.60
B           5   3  4.25         5   3  4.25           5   4  4.75

Разумеется, списки могут быть разными для разных признаков, например:

In [24]:
instruction = {
    'Алгебра': ['max', 'min', 'mean'],
    'Геометрия': ['max', 'min', 'mean'],
    'Информатика': ['max', 'min'],
   
}

result = df.groupby('Класс').agg(instruction)
display(result)

Алгебра           Геометрия           Информатика    
          max min  mean       max min  mean         max min
Класс                                                      
A           5   4  4.60         5   4  4.60           5   3
B           5   3  4.25         5   3  4.25           5   4

И так далее. Фактически, мы уже сейчас можем делать почти все что угодно: задавать признаки для группировки, указывать признаки для агрегации и агрегирующие функции.

Но что, если мы захотим вывести какую-то свою, собственноручно изготовленную функцию, которая не входит в перечень стандартных функций для агрегации? Да не вопрос! Мы можем написать собственную функцию.

Например, напишем функцию для разброса значений и применим ее к оценкам по информатике:

In [25]:
def spread_of_valies(series):
    return series.max() - series.min()

instruction = {
    'Алгебра': ['max', 'min'],
    'Геометрия': ['max', 'min', 'mean'],
    'Информатика': spread_of_valies,
   
}

result = df.groupby('Класс').agg(instruction)
display(result)

Алгебра     Геометрия                Информатика
          max min       max min  mean spread_of_valies
Класс                                                 
A           5   4         5   4  4.60                2
B           5   3         5   3  4.25                1

Теперь мы уже можем делать все что угодно? Нет, это еще не все. Осталась одна маленькая деталь.

Допустим, нам не нравится шапка вывода сводной таблицы (так, в примере выше она у нас двухуровневая, а это бывает весьма неудобно). У метода `agg` есть возможность преодолеть и эту неприятность. Мы можем использовать не словарь-инструкцию, а именованные аргументы метода:

In [26]:
result = df.groupby('Класс').agg(
    Алгебра_min = ('Алгебра', 'min'),
    Информатика_std = ('Информатика', 'std'),
    Геометрия_spread = ('Геометрия', spread_of_valies)
)

display(result)

,Алгебра_min,Информатика_std,Геометрия_spread
Класс,,,
A,4,0.894427,1
B,3,0.500000,2


При таком подходе в качестве значения аргумента указывается кортеж (имя признака, агрегирующая функция), имена аргументов становятся названиями столбцов результирующего датафрейма, и мы можем многоуровневую шапку таблицы перевести в более удобный одноуровневый вариант. 

Например, так:

In [27]:
result = df.groupby('Класс').agg(
    Алгебра_min = ('Алгебра', 'min'),
    Алгебра_max = ('Алгебра', 'max'),
    Информатика_std = ('Информатика', 'std'),
    Геометрия_spread = ('Геометрия', spread_of_valies)
)

display(result)

,Алгебра_min,Алгебра_max,Информатика_std,Геометрия_spread
Класс,,,,
A,4,5,0.894427,1
B,3,5,0.500000,2


Метод `agg` с именованными аргументами и  пользовательскими функциями делает нас неуязвимым. Нам не важно, есть ли в Pandas встроенная функция для нашей задачи, мы просто говорим: «Сделай вот по этой формуле и отобрази вот в таком виде», — и он делает.

Вот теперь уже точно все. Теперь мы можем делать с данными все что угодно.

### Синтаксический сахар

Иногда нам не нужно так много мощи. Иногда мы просто хотим быстро узнать средний балл по классам. Для этого добрые разработчики придумали сокращения. Вместо того чтобы строить словарь, мы можем написать:

In [28]:
df.groupby('Класс')['Алгебра'].mean()

Класс
A    4.60
B    4.25
Name: Алгебра, dtype: float64

Да, мы можем так поступить. Но, во-первых, никто не говорил, что мы должны так поступать. Во-вторых, обратите внимание, насколько позорно выглядит выход этой ячейки: какая-то жалкая серия, которая не выдерживает никакого сравнения с нашими осмысленными и изящными датафреймами, которые мы получали выше. А в третьих, сахар — для слабаков. 

Настоящий датасайентист всегда идет по пути джедая: он использует метод `agg` и не разменивается на мелочи.

## § 4. Метод `transform` в связке с `groupby`

Метод `transform` не является исключительной прерогативой `groupby`. Он используется для применения какой-либо функции к  датафреймам или сериям с сохранением исходной структуры данных (формы и индексов). Это делает его похожим на метод `apply`, но сохрание формы и индексов — это обязателдьное условие.

### Метод `transform`

Для начала (просто, чтобы посмотреть, как это работает) попробуем повысить успеваемость всех наших школьников сразу на 5 баллов (нам не жалко, можем и на 15 повысить):

In [29]:
def function(series):
    return series+5

df[df.columns[2:-2]].transform(function)

,Алгебра,Геометрия,Информатика,Литература,История
0,10,10,9,8,9
1,8,10,10,10,8
2,10,10,10,9,10
3,9,10,8,7,8
4,10,9,8,8,9
5,9,10,10,9,9
6,9,9,8,8,8
7,10,9,9,8,9
8,10,8,10,8,9


Разумеется, получилась таблица, лишенная всякого смысла, потому что таких оценок не бывает, но пока мы просто пробуем метод «на ощупь»: мы взяли датафрейм `df`, выкинули первые два и последние два столбца в силу того, что по своей семантике это никакие не оценки, а строковые характеристики наших данных (сделали срез по  столбцам`df[df.columns[2:-2]]`), а к тому, что получилось применили `transform` с определенной выше функцией `function` в качестве аргумента.

Раз все работает, то наша следующая мысль состоит в том, что а давайте каждый столбец заменим на среднее значение по столбцу. В этом тоже нет никакого смысла, но это нас тоже не сильно беспокоит: мы продолжаем пробовать метод «на ощупь». Однако на этот раз нас ожидает фиаско:

In [30]:
# df[df.columns[2:-2]].transform('mean')

Что не так? 

Не так то, что мы нарушили структуру данных. Мы сказали методу `transform`: «Возьми каждый столбец, вычисли его среднее значение и замени все элементы столбца этим одним числом».

Но `transform` рассуждает иначе: «Вы дали мне на вход `'mean'`. Я, конечно, все понимаю: это стандартная функция и все такое. Но для каждой колонки `mean` — это функция, которая возвращает одно число. Иначе говоря, скаляр. А я, метод `transform`, ожидаю, что функция вернет столько же элементов, сколько вошло. Иначе говоря, вектор (ну или серию, если уж быть совсем точным). Ваш результат не совпадает по длине с исходным столбцом. Это фиаско, коллеги».

Метод `transform` обязательно должен вернуть объект той же формы, что и исходный. Это его железобетонный догмат, а догматы следует соблюдать. Соблюсти его мы сможем, например, так:

In [31]:
def function(series):
    series_bufer = pd.Series(np.ones(len(series)))
    series_bufer.index = series.index
    series_bufer = series_bufer * series.mean()
    return series_bufer

df[df.columns[2:-2]].transform(function)

,Алгебра,Геометрия,Информатика,Литература,История
0,4.444444,4.444444,4.111111,3.333333,3.777778
1,4.444444,4.444444,4.111111,3.333333,3.777778
2,4.444444,4.444444,4.111111,3.333333,3.777778
3,4.444444,4.444444,4.111111,3.333333,3.777778
4,4.444444,4.444444,4.111111,3.333333,3.777778
5,4.444444,4.444444,4.111111,3.333333,3.777778
6,4.444444,4.444444,4.111111,3.333333,3.777778
7,4.444444,4.444444,4.111111,3.333333,3.777778
8,4.444444,4.444444,4.111111,3.333333,3.777778


Теперь все получилось: каждый столбец заменен на константное значение среднего по столбцу. Мы предупреждали, что эта операция абсолютно бессмыслена. Зато ее можно применить не только к столбцам, но и к строкам, если указать `axis=1`:

In [32]:
df[df.columns[2:-2]].transform(function, axis=1)

,Алгебра,Геометрия,Информатика,Литература,История
0,4.2,4.2,4.2,4.2,4.2
1,4.2,4.2,4.2,4.2,4.2
2,4.8,4.8,4.8,4.8,4.8
3,3.4,3.4,3.4,3.4,3.4
4,3.8,3.8,3.8,3.8,3.8
5,4.4,4.4,4.4,4.4,4.4
6,3.4,3.4,3.4,3.4,3.4
7,4.0,4.0,4.0,4.0,4.0
8,4.0,4.0,4.0,4.0,4.0


### А при чем тут `groupby`?

А при том, что `groupby` — это лучший друг `transform`, который помогает ему соблюдать его «железобетонный догмат» о сохранении формы данных, не прибегая к написанию страшных функций с `np.ones` и ручным указанием индексов.

Давайте выведем наши данные по успеваемости (чтобы их было видно):

In [33]:
df

,ID,Имя,Алгебра,Геометрия,Информатика,Литература,История,Класс,Пол
0,A33,Андрей,5,5,4,3,4,A,М
1,B15,Серёжа,3,5,5,5,3,B,М
2,A95,Настя,5,5,5,4,5,A,Ж
3,A56,Серёжа,4,5,3,2,3,A,М
4,A32,Дима,5,4,3,3,4,A,М
5,B56,Настя,4,5,5,4,4,B,Ж
6,A41,Миша,4,4,3,3,3,A,М
7,B14,Катя,5,4,4,3,4,B,Ж
8,B77,Дима,5,3,5,3,4,B,М


И вычислим среднюю оценку по алгебре для ребят с совпадающими именами. Это не самая разумная с точки зрения педагогики операция, но и не самая бессмысленная. Она нам нужна, потому что сейчас мы пройдемся по всем внутринним оперциям вместе с методом `transform`, и при таком разбиении на группы (в группах окажется либо один ученик, либо максимум — два) нам будет нетрудно в уме посчитать средние.

Вот что возвращает метод `transform` (заметьте, что мы сначала применили группировку по имени ко всему датафрему, а потом — трансформацию по среднему к одному отдельно взятому столбцу `'Алгебра'`):

In [34]:
df.groupby('Имя')['Алгебра'].transform('mean')

0    5.0
1    3.5
2    4.5
3    3.5
4    5.0
5    4.5
6    4.0
7    5.0
8    5.0
Name: Алгебра, dtype: float64

Итак, после применения `groupby` метод `transform` работает под капотом внутри групп:

- он видит, что Андрей один. Поэтому его среднее по алгебре совпадает с его оценкой (это 5), и он пишет эту оценку на первое место: туда, где расположен Андрей;

- дальше он видит, что Сереж двое, у одного 3 по алгебре, у другого — 4 (среднее 3.5), и он два раза учитывает их среднюю оценку на местах, где они расположены, это 1 и 3 позиции в серии;

- дальше видит, что есть две девочки по имени Настя, у одной 5 по алгебре, у другой — 4 (среднее 4.5), и два раза учитывает их среднюю оценку на местах, где они расположены (2 и 5 позиции в серии);

- дальше видит, что есть два мальчика по имени Дима, и поступает с ними точно так же, как уже поступил с Срежами и Настями;

- ну и наконец, видит, что есть один Миша и одна Катя, и поступает сними точно так же, как уже поступил с Сережей.

В результате получается серия той же длины, что и была, с тем же самым индексом, что и был.

Мы можем применить `transform` для вычисления средних по классам для всех предметов гуманитарного цикла (а это, в отличие от предыдущих, уже совершенно осмысленная с педагогической точки зрения операция):

In [35]:
df.groupby('Класс')[['Литература', 'История']].transform('mean')

,Литература,История
0,3.00,3.80
1,3.75,3.75
2,3.00,3.80
3,3.00,3.80
4,3.00,3.80
5,3.75,3.75
6,3.00,3.80
7,3.75,3.75
8,3.75,3.75


Поскольку у нас всего два класса, в каждой колонке получилось по два занчения и они распределены строго по классам: A и B (если интересно, можете проверить по индексам). Вывод сразу нескольких столбцов с применением одной, общей для всех агрегирующей функции — это довольно удобно. Но что делать, если в силу каких-то причин нам по одной группировке для разных признаков понадобятся разные агрегирующие функции? 

Самый надежный путь — применить `groupby` ко всему датафрейму (что естественно), а потом применять  `transform`  к каждому столбцу по отдельности (что, вообще говоря, противоестественно, но `transform`, в отличие от `agg` — парень простой, он словарей не понимает):

In [36]:
algebra_ave = df.groupby('Класс')['Алгебра'].transform('mean')
informatics_std = df.groupby('Класс')['Информатика'].transform('std')
display(algebra_ave)
display(informatics_std)

0    4.60
1    4.25
2    4.60
3    4.60
4    4.60
5    4.25
6    4.60
7    4.25
8    4.25
Name: Алгебра, dtype: float64

0    0.894427
1    0.500000
2    0.894427
3    0.894427
4    0.894427
5    0.500000
6    0.894427
7    0.500000
8    0.500000
Name: Информатика, dtype: float64

И, кстати, по поводу `agg`. Продолжая метафору симпатии, дружбы и всего такого, можно сказать, что, если `groupby` — лучший друг для `transform`, то `agg` — это его злейший враг. Они антогонисты: `agg`сжимает данные, `transform` сохраняет их полный формат, в силу чего применить их одновременно нельзя даже теоретически.

## § 5. Интеграция в Data Science

Теперь, когда мы узнали достаточно много, мы можем разобрать вполне себе серьезный (а не игрушечный, как выше) пример.

Грузим данные из файла:

In [37]:
df_hr = pd.read_csv('data/hr.csv')
df_hr

,ID,Отдел,Город,Зарплата,KPI_score,Стаж,Оценка_менеджера
0,EMP_0001,HR,Новосибирск,217340,62,1,3.1
1,EMP_0002,Финансы,СПб,217860,38,10,2.9
2,EMP_0003,IT,Новосибирск,242121,67,5,2.1
3,EMP_0004,Финансы,Казань,112493,38,12,3.5
4,EMP_0005,Финансы,Москва,86027,67,6,3.2
...,...,...,...,...,...,...,...
1495,EMP_1496,HR,Новосибирск,99522,19,5,2.8
1496,EMP_1497,Финансы,Екатеринбург,185958,18,10,3.9
1497,EMP_1498,Финансы,Казань,161249,95,9,4.6
1498,EMP_1499,HR,СПб,57485,30,12,2.1


Сначала мы хотим посмотреть общую картину: среднюю зарплату и средний KPI по отделам. Тут нам поможет старый добрый `agg`, потому что нам нужен отчет-сводка:

In [38]:
report = df_hr.groupby('Отдел').agg({
    'Зарплата': 'mean',
    'KPI_score': 'mean'
})
round(report, 2)

,Зарплата,KPI_score
Отдел,,
HR,145586.02,48.80
IT,146558.73,51.06
Маркетинг,149566.97,50.86
Продажи,152410.89,52.19
Финансы,152818.32,50.36


Дальше переходим к зарплатная географии: в каких городах самые высокие зарплаты, или кому на Руси жить хорошо? Группируем по городам и ищем максимум по среднему.

In [39]:
instruction = {'Зарплата': 'mean'}

city_salary = df_hr.groupby('Город').agg(instruction).sort_values(by = 'Зарплата', ascending=False)
round(city_salary, 2)

,Зарплата
Город,
СПб,151820.16
Казань,151616.67
Москва,149225.27
Новосибирск,148433.93
Екатеринбург,146787.38


Разброс зарплат — где «социальная несправедливость» выше? Ищем отдел, где зарплаты прыгают сильнее всего.

In [40]:
instruction = {'Зарплата': 'std'}
salary_std = df_hr.groupby('Отдел').agg(instruction).sort_values(by = 'Зарплата', ascending=False)

salary_std

,Зарплата
Отдел,
Маркетинг,60493.574809
Продажи,58192.998817
Финансы,57248.109293
IT,57066.797724
HR,55980.186929


И, наконец, смертельный номер: нам нужно вписать каждого сотрудника в контекст его отдела. Мы хотим добавить столбец, который покажет, недоплачивают человеку по сравнению с коллегами по отделу или наоборот — переплачивают.

In [41]:
df_hr['Средняя_отдела'] = df_hr.groupby('Отдел')['Зарплата'].transform('mean')
df_hr['Отклонение_от_среднего'] = df_hr['Зарплата'] - df_hr['Средняя_отдела']
df_hr.sort_values('Отклонение_от_среднего', ascending=False).head()

,ID,Отдел,Город,Зарплата,KPI_score,Стаж,Оценка_менеджера,Средняя_отдела,Отклонение_от_среднего
1104,EMP_1105,IT,Новосибирск,249570,27,10,4.5,146558.726962,103011.273038
875,EMP_0876,HR,Екатеринбург,248522,9,12,3.4,145586.017483,102935.982517
279,EMP_0280,IT,Новосибирск,248700,60,7,4.2,146558.726962,102141.273038
606,EMP_0607,HR,Казань,247724,53,13,4.9,145586.017483,102137.982517
154,EMP_0155,IT,Новосибирск,248568,5,12,3.1,146558.726962,102009.273038


# Домашнее задание

Прочитайте данные о продажах в трех  магазинах из файла `df_shop.csv`.

In [42]:
# Код

>#### Задание 
>1) Разделите данные по магазинам, используя для агрегации число совершенных покупок в каждом магазине.
>2) Для каждого магазина вычислите: средний чек и его отклонение.
>3) Уточните вывод: для суммы укажите полный разброс (min, mean и max), для количества товаров — среднее, для часа — тоже среднее.
>4) Для каждого магазина вычислите отклонение каждой покупки от среднего чека по магазину.
>5) Отфильтруйте только те чеки, где отклонение превышает k стандартных отклонения по магазину (k — варьируемый параметр).